# استخراج النصوص من المخطوطات العربية
ارفع ملف PDF وسيتم استخراج النص العربي الكامل مع التشكيل والحركات وتصديره كملفات `.docx` و `.txt`.

In [ ]:
# === 1. تثبيت المتطلبات ===
!apt-get -qq update && apt-get -qq install -y poppler-utils
!pip install -q transformers accelerate pdf2image python-docx torchvision torch qwen-vl-utils

# === 2. تحميل النموذج ===
import torch
from transformers import Qwen2VLForConditionalGeneration, AutoProcessor

MODEL_ID = "Qwen/Qwen2-VL-2B-Instruct"

model = Qwen2VLForConditionalGeneration.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    device_map="auto",
)
processor = AutoProcessor.from_pretrained(MODEL_ID)

print("\u2705 تم تحميل النموذج بنجاح")

# === 3. رفع ملف PDF ===
from google.colab import files as colab_files

uploaded = colab_files.upload()
pdf_path = list(uploaded.keys())[0]
print(f"\u2705 تم رفع: {pdf_path}")

# === 4. تحويل PDF إلى صور ومعالجة كل صفحة ===
from pdf2image import convert_from_path
from qwen_vl_utils import process_vision_info

pages = convert_from_path(pdf_path, dpi=180)
total_pages = len(pages)
all_texts = []

SYSTEM_PROMPT = (
    "استخرج النص العربي الكامل من الصورة بدقة متناهية "
    "مع الحفاظ الصارم على علامات الترقيم، والتشكيل والحركات، "
    "دون أي تعليق جانبي أو مقدمات."
)

for idx, page_img in enumerate(pages, 1):
    print(f"معالجة صفحة ({idx} / {total_pages})...", end="\r")

    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image", "image": page_img},
                {"type": "text", "text": SYSTEM_PROMPT},
            ],
        }
    ]

    text_prompt = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    image_inputs, video_inputs = process_vision_info(messages)

    inputs = processor(
        text=[text_prompt],
        images=image_inputs,
        videos=video_inputs,
        padding=True,
        return_tensors="pt",
    ).to(model.device)

    with torch.no_grad():
        output_ids = model.generate(**inputs, max_new_tokens=2048)

    generated_ids = output_ids[:, inputs.input_ids.shape[1]:]
    result = processor.batch_decode(generated_ids, skip_special_tokens=True)[0].strip()
    all_texts.append(result)

print(f"\n\u2705 تمت معالجة {total_pages} صفحة بنجاح")

# === 5. تصدير النتائج وتنزيلها ===
import os
from docx import Document

base_name = os.path.splitext(pdf_path)[0]
docx_path = f"{base_name}_نتيجة.docx"
txt_path = f"{base_name}_نتيجة.txt"

# Word document
doc = Document()
for i, text in enumerate(all_texts, 1):
    doc.add_heading(f"صفحة {i}", level=2)
    doc.add_paragraph(text)
doc.save(docx_path)

# Plain text
with open(txt_path, "w", encoding="utf-8") as f:
    for i, text in enumerate(all_texts, 1):
        f.write(f"--- صفحة {i} ---\n{text}\n\n")

# Auto-download
colab_files.download(docx_path)
colab_files.download(txt_path)

print(f"\u2705 تم تصدير: {docx_path}")
print(f"\u2705 تم تصدير: {txt_path}")